# OUEM D4a · research v5 one-update Satlas smoke
Use a fresh Colab runtime after separate approval under Issue #88. Historical T4 use does not guarantee current allocation, lifetime, quota or cost. Verify the current Colab terms, GPU availability and private Drive access in the UI. Review the provisional 900-second and 12 GiB reserved-memory ceilings with the run owner. The runtime and dependencies have not yet been tested for v5.


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib, importlib.metadata, json, os, shutil, subprocess, sys, tempfile, zipfile

MOUNT_DRIVE = False
STAGE_INPUTS = False
RUN_PREFLIGHT = False
RUN_SMOKE = False
RUN_RESTORE = False
PERSIST_EVIDENCE = False
COMMIT = "30c3d9a63543550bfe7321f5a7dad11e646e3b94"
REPO_URL = "https://github.com/iwamoto31206803-max/ouem-canopy-segmentation.git"
PROJECT = Path("/content/drive/MyDrive") / "<private-project-folder>"
ZIP = PROJECT / "02_datasets" / "ouem-research-v5.zip"
ZIP_SHA = "<SHA-256 from private local receipt>"
RUN_ID = "d4a-v5-smoke-01"
SECONDS = 900  # Provisional. Confirm before enabling execution.
VRAM_BYTES = 12 * 1024**3  # Provisional reserved VRAM ceiling.
WORK = Path("/content") / RUN_ID
REPO, STAGE, OUT = (WORK / name for name in ("repo", "inputs", "out"))
BACKUP = PROJECT / "03_checkpoints" / RUN_ID
RESULTS = PROJECT / "04_results" / RUN_ID
WEIGHT = PROJECT / "01_pretrained" / "aerial_swinb_si.pth"
PRETRAINED = "f8abf21cadaab2a71859e9d172bec4e5d6b7cc9710882a2f6a4d8aae826db036"
PINS = {
    "dataset-v5/crop_plan.json": "b6711c33eafed51dd8d42040abf13e3a52601d5be54b08479217ac7e043c8692",
    "research_review.json": "ccda53e9dac2c40fda6be718b9fc8a57dc35e6210fcc271a5a5533e390456dbc",
    "dataset_freeze_v5.json": "d7a72bdf1c792678cee0030c72dded18d3cdd6639a800c12ebafc63ba3fa1978",
    "candidates.json": "bf71faf1ab512a326804f808aeff541f8e1363c8dece6d2250c6ebe7cbece0db",
}
IDENTITY = "a645e831c4214cab913506e01fe9160f5272cf8aed1a6b00bbbf35964cd3b6d7"

def sha(path):
    with path.open("rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest()

def require_hash(path, expected):
    if sha(path) != expected:
        raise ValueError(f"SHA mismatch: {path.name}")

def require_fresh(path):
    if path.exists() or path.is_symlink():
        raise FileExistsError(path)

def copy_verified(source, destination, expected):
    require_hash(source, expected)
    with source.open("rb") as src, destination.open("xb") as dst:
        shutil.copyfileobj(src, dst)
    require_hash(destination, expected)

def extract_v5(archive, destination, expected):
    require_hash(archive, expected)
    require_fresh(destination)
    with zipfile.ZipFile(archive) as z:
        entries = z.infolist()
        names = [entry.filename for entry in entries]
        if len(names) != 475 or len(set(names)) != 475 or sum(e.file_size for e in entries) != 23090220:
            raise ValueError("Unexpected ZIP inventory or size")
        for name in names:
            path = PurePosixPath(name)
            if (path.is_absolute() or path.as_posix() != name or ".." in path.parts
                    or "\\" in name or ":" in name):
                raise ValueError("Unsafe ZIP member")
        for name, pin in PINS.items():
            if hashlib.sha256(z.read(name)).hexdigest() != pin:
                raise ValueError(f"Contract SHA mismatch: {name}")
        plan = json.loads(z.read("dataset-v5/crop_plan.json"))
        products = {"dataset-v5/" + p["path"]: p["sha256"]
                    for sample in plan["samples"] for p in sample["products"].values()}
        if len(products) != 471 or set(names) != set(PINS) | set(products):
            raise ValueError("ZIP does not match the pinned plan")
        destination.mkdir()
        for name, pin in (PINS | products).items():
            target = destination / name
            target.parent.mkdir(parents=True, exist_ok=True)
            with z.open(name) as src, target.open("xb") as dst:
                shutil.copyfileobj(src, dst)
            require_hash(target, pin)
    require_hash(archive, expected)


Mount only after the project owner approves access. Set the project path and archive SHA from the private receipt. Confirm that the ZIP already exists on Drive before staging. If a dependency is missing, resolve the version in the authorized runtime first. Keep the existing CUDA Torch/torchvision installation. If SatlasPretrain is missing, inspect the proposed dependency changes first (e.g. with pip install --dry-run satlaspretrain-models==0.3.1). Recheck imports after any authorized install.

In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    if not PROJECT.is_dir():
        raise FileNotFoundError("Private project folder is missing")


In [ ]:
import numpy, rasterio, torch, torchvision
from PIL import Image
import satlaspretrain_models
if not torch.cuda.is_available():
    raise RuntimeError("CUDA device required")
satlas = importlib.metadata.version("satlaspretrain-models")
gpu = torch.cuda.get_device_properties(0)
ENVIRONMENT = dict(python=sys.version, torch=torch.__version__, torchvision=torchvision.__version__,
                   cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version(), satlas=satlas,
                   gpu=gpu.name, total_vram=gpu.total_memory, numpy=numpy.__version__,
                   rasterio=rasterio.__version__, gdal=rasterio.__gdal_version__, pillow=Image.__version__)
print(ENVIRONMENT)
if satlas != "0.3.1" or not 0 < VRAM_BYTES < gpu.total_memory or SECONDS <= 0:
    raise ValueError("Package version or proposed ceilings incompatible with this runtime")


Enable staging only after data-transfer permission and the owner-approved private Drive location are confirmed. A failed stage leaves its partial scratch directory for diagnosis. Use a fresh run path for a retry. Reserve at least 4 GiB on scratch and 3 GiB on Drive for checkpoint plus result evidence. These are planning allowances. Confirm Drive quota in the UI if the mount cannot report it.

In [ ]:
if STAGE_INPUTS:
    import re
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,63}", RUN_ID):
        raise ValueError("Invalid run ID")
    for path in (WORK, BACKUP, RESULTS):
        require_fresh(path)
    if not PROJECT.is_dir():
        raise FileNotFoundError("Private project folder is missing")
    if shutil.disk_usage("/content").free < 4 * 1024**3:
        raise OSError("Insufficient scratch storage")
    require_hash(ZIP, ZIP_SHA)
    require_hash(WEIGHT, PRETRAINED)
    with tempfile.TemporaryDirectory(dir="/content") as tmp:
        probe = Path(tmp) / "a"
        probe.write_bytes(b"hardlink probe")
        link = Path(tmp) / "b"
        os.link(probe, link)
        if not os.path.samefile(probe, link):
            raise OSError("Scratch hardlink probe failed")
    WORK.mkdir()
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
    subprocess.run(["git", "checkout", "--detach", COMMIT], cwd=REPO, check=True)
    extract_v5(ZIP, STAGE, ZIP_SHA)
    copy_verified(WEIGHT, STAGE / WEIGHT.name, PRETRAINED)


In [ ]:
ARGS = [sys.executable, "-B", "scripts/research_cuda.py",
        "--dataset", str(STAGE / "dataset-v5"), "--review", str(STAGE / "research_review.json"),
        "--freeze", str(STAGE / "dataset_freeze_v5.json"), "--candidates", str(STAGE / "candidates.json"),
        "--initializer", str(STAGE / WEIGHT.name), "--output", str(OUT), "--backup", str(BACKUP),
        "--seconds", str(SECONDS), "--vram-bytes", str(VRAM_BYTES), "--run-id", RUN_ID, "--smoke"]

def check_checkout():
    head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
    status = subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True)
    if head != COMMIT or status.strip():
        raise ValueError("Expected clean checkout at pinned commit")

def cli(*extra):
    check_checkout()
    return subprocess.run(ARGS + list(extra), cwd=REPO, text=True, check=True,
                          stdout=subprocess.PIPE, stderr=None).stdout

PREFLIGHT_ARGS = None
if RUN_PREFLIGHT:
    import ast
    for path in (OUT, BACKUP, RESULTS):
        require_fresh(path)
    for parent in (BACKUP.parent, RESULTS.parent):
        if not parent.is_dir() or not os.access(parent, os.W_OK):
            raise PermissionError("Private Drive output parent unavailable")
    if shutil.disk_usage(BACKUP.parent).free < 3 * 1024**3:
        raise OSError("Insufficient Drive storage")
    preflight = ast.literal_eval(cli().strip().splitlines()[-1])
    from hashlib import sha256
    identity = preflight["binding"]["inputs"]
    canonical = (json.dumps(identity, sort_keys=True, indent=2, ensure_ascii=True) + "\n").encode()
    if preflight["status"] != "PREFLIGHT_PASS" or sha256(canonical).hexdigest() != IDENTITY:
        raise ValueError("Pinned v5 preflight failed")
    print(preflight)
    PREFLIGHT_ARGS = tuple(ARGS)


Run the smoke cell only after explicit approval of the actual GPU, runtime limits, storage and private transfer. The smoke evaluates Val18 at steps 0 and 1 and then exits. Failures require review. Preserve available evidence before closing the session.

In [ ]:
if RUN_SMOKE:
    if PREFLIGHT_ARGS != tuple(ARGS):
        raise RuntimeError("Run preflight with these exact arguments first")
    print(cli("--execute"))


In [ ]:
def completed_smoke():
    result = json.loads((OUT / "result.json").read_text())
    if result["status"] != "COMPLETE" or result["completed_steps"] != 1:
        raise ValueError("Expected completed one-update smoke")
    for step in (0, 1):
        metrics = json.loads((OUT / f"val_{step:03d}" / "metrics.json").read_text())
        if len(metrics["windows"]) != 18:
            raise ValueError("Expected fixed Val18 metrics")
    copy = json.loads((OUT / "backup_001.json").read_text())
    checkpoint = Path(copy["destination_directory"]) / copy["filename"]
    receipt = json.loads((BACKUP / "step_001" / "runtime.json").read_text())
    if (not copy["copy_sha_match"] or checkpoint != BACKUP / "step_001" / "checkpoint_001.pth"
            or receipt["checkpoint_sha256"] != copy["sha256"] or receipt["segment_id"] != result["segment_id"]):
        raise ValueError("Backup receipt mismatch")
    require_hash(checkpoint, copy["sha256"])
    require_hash(OUT / "checkpoint_001.pth", copy["sha256"])
    return result, checkpoint, copy["sha256"]

if RUN_RESTORE:
    import ast
    result, checkpoint, checkpoint_sha = completed_smoke()
    restored = ast.literal_eval(cli("--verify-restore", "--resume", str(checkpoint),
                                    "--resume-sha256", checkpoint_sha, "--lineage-id",
                                    result["binding"]["lineage_id"]).strip().splitlines()[-1])
    if restored["status"] != "RESTORE_PASS" or restored["segment_id"] != result["segment_id"]:
        raise ValueError("Separate-process restore failed")
    with (OUT / "restore.json").open("x") as f:
        json.dump(restored, f, indent=2)


In [ ]:
if PERSIST_EVIDENCE:
    if not OUT.is_dir():
        raise FileNotFoundError("Run output missing")
    require_fresh(RESULTS)
    RESULTS.mkdir()
    hashes = {}
    for source in sorted(OUT.rglob("*")):
        if not source.is_file() or source.suffix not in (".json", ".png"):
            continue
        relative = source.relative_to(OUT)
        destination = RESULTS / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        hashes[relative.as_posix()] = sha(source)
        copy_verified(source, destination, hashes[relative.as_posix()])
    with (RESULTS / "sha256.json").open("x") as f:
        json.dump(hashes, f, indent=2)
    print(f"Verified {len(hashes)} evidence files. Checkpoints are in the separate backup folder.")


Before closing Colab, confirm the checkpoint, runtime receipt and evidence SHA index on Drive. The currently obtainable CUDA runtime, package compatibility, Drive permissions, actual transfer and runtime cost, latency and reserved VRAM remain to be measured.